<div style="background: linear-gradient(135deg, #0f172a, #1e293b); color:white; padding:25px; border-radius:10px;
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;">Market Sentiment Analysis from Tweets</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#94a3b8;">
    Final Pipeline - DeBERTa-v3 Fine-Tuned
  </h3>

  <hr style="width:60%; border:1px solid #475569; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Text Mining (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#cbd5e1;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e2e8f0; padding:15px 20px; border-left:5px solid #475569;
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">
  <b>Notebook Description</b><br>
  Single ready-to-run pipeline that fine-tunes DeBERTa-v3 on the full training corpus (9,543 tweets) and generates <code>pred_24.csv</code>. Model selection and hyperparameter decisions are documented in the experiments notebook (tm_tests_24).
</div>

<div style="background-color:#0f172a; color:#e0f2fe; padding:15px 20px; border-left:5px solid #38bdf8;
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">

  <b>Important Note</b><br>
  This notebook was developed and executed on <code>Google Colab</code>. If you intend to run it locally, the following changes must be made before execution:
  <ul style="margin-top:8px; margin-bottom:0;">
    <li>Change the DataLoader parameter <code>num_workers</code> from <code>2</code> to <code>0</code>. During our tests on Apple devices, the pipeline only executed successfully with this configuration. This is because Google Colab supports multiprocessing more reliably, while some local devices may not.</li>
    <li>Remove the Google Drive mounting cell (<code>drive.mount('/content/drive')</code>) and update all file paths to match your local directory structure.</li>
  </ul>

</div>

**<h3>Table of Contents</h3>**
* [1. Environment Setup](#1-setup)
* [2. Data](#2-data)
* [3. Configuration](#3-config)
* [4. Model Components](#4-components)
* [5. Fine-Tuning](#5-finetune)
* [6. Test Set Predictions](#6-predict)
* [7. Save Predictions](#7-save)

<div id="1-setup" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">1. Environment Setup</h2>
</div>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Text-Mining'
os.chdir(f'{PROJECT_PATH}/group_24')
print(f'Working dir: {os.getcwd()}')

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Text-Mining/group_24


In [2]:
!pip install -q transformers accelerate sentencepiece

In [3]:
import warnings, os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.utils.class_weight import compute_class_weight

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LABEL_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : Tesla T4


<div id = "2-data" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">2. Data</h2>
</div>

Transformer models operate on raw tweet text - the tokeniser handles subword segmentation internally. The domain-aware preprocessing pipeline (sentinel tokens, stopword removal, lemmatisation) developed in the experiments notebook was applied only to the classical ML feature representations.

In [4]:
train_df = pd.read_csv('../data/train.csv')
test_df  = pd.read_csv('../data/test.csv')

x_train  = train_df['text'].tolist()
y_train  = train_df['label'].values
x_test   = test_df['text'].tolist()
test_ids = test_df['id'].values

print(f'Train : {len(x_train):,} tweets')
print(f'Test  : {len(x_test):,}  tweets')
print(f'Labels: {pd.Series(y_train).map(LABEL_NAMES).value_counts().to_dict()}')

class_weights = torch.tensor(
    compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train),
    dtype=torch.float32
).to(DEVICE)
print(f'\nClass weights: Bearish={class_weights[0]:.3f}, Bullish={class_weights[1]:.3f}, Neutral={class_weights[2]:.3f}')

Train : 9,543 tweets
Test  : 2,388  tweets
Labels: {'Neutral': 6178, 'Bullish': 1923, 'Bearish': 1442}

Class weights: Bearish=2.206, Bullish=1.654, Neutral=0.515


<div id = "3-config" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">3. Configuration</h2>
</div>

In [5]:
class Config:
    MODEL_NAME    = 'nickmuchi/deberta-v3-base-finetuned-finance-text-classification'
    MAX_LENGTH    = 128
    BATCH_SIZE    = 16
    LEARNING_RATE = 1e-5
    NUM_EPOCHS    = 17
    WARMUP_RATIO  = 0.20
    WEIGHT_DECAY  = 0.01
    LLRD_DECAY    = 0.9
    LABEL_SMOOTH  = 0.05

cfg = Config()
print(f'Model      : {cfg.MODEL_NAME}')
print(f'Epochs     : {cfg.NUM_EPOCHS}')
print(f'LR         : {cfg.LEARNING_RATE}')
print(f'Batch      : {cfg.BATCH_SIZE}')
print(f'LLRD decay : {cfg.LLRD_DECAY}')

Model      : nickmuchi/deberta-v3-base-finetuned-finance-text-classification
Epochs     : 17
LR         : 1e-05
Batch      : 16
LLRD decay : 0.9


<div id = "4-components" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">4. Model Components</h2>
</div>

In [6]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt'
        )
        item = {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def get_param_groups(model, base_lr, decay, weight_decay):
    """Layer-wise learning rate decay — lower LR for earlier layers."""
    no_decay = ['bias', 'LayerNorm.weight']
    n_layers  = model.config.num_hidden_layers
    assigned  = set()
    groups    = []

    def add_group(params_list, lr):
        wd    = [p for n, p in params_list if not any(nd in n for nd in no_decay)]
        no_wd = [p for n, p in params_list if     any(nd in n for nd in no_decay)]
        if wd:    groups.append({'params': wd,    'lr': lr, 'weight_decay': weight_decay})
        if no_wd: groups.append({'params': no_wd, 'lr': lr, 'weight_decay': 0.0})
        for n, _ in params_list: assigned.add(n)

    add_group([(n, p) for n, p in model.named_parameters()
               if any(x in n for x in ['classifier', 'pooler'])], base_lr)

    for i in range(n_layers - 1, -1, -1):
        layer_lr = base_lr * (decay ** (n_layers - 1 - i))
        add_group([(n, p) for n, p in model.named_parameters()
                   if f'layer.{i}.' in n and n not in assigned], layer_lr)

    add_group([(n, p) for n, p in model.named_parameters()
               if n not in assigned], base_lr * (decay ** n_layers))
    return groups

<div id = "5-finetune" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">5. Fine-Tuning</h2>
</div>

The full training corpus (9,543 tweets) is used -> no validation split is held out at this stage since model selection was performed during experiments. Training runs for a fixed 17 epochs (best was 12, the validation part was added to training, so it is fair to add 5 more epochs) with LLRD, weighted cross-entropy, and label smoothing.

In [7]:
print(f'Loading {cfg.MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    cfg.MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True
).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Loading nickmuchi/deberta-v3-base-finetuned-finance-text-classification ...


config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Parameters: 184,424,451


In [8]:
train_ds     = TweetDataset(x_train, y_train, tokenizer, cfg.MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)

total_steps  = len(train_loader) * cfg.NUM_EPOCHS
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)

optimizer = AdamW(get_param_groups(model, cfg.LEARNING_RATE, cfg.LLRD_DECAY, cfg.WEIGHT_DECAY))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn   = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.LABEL_SMOOTH)

print(f'Steps/epoch : {len(train_loader)}')
print(f'Total steps : {total_steps}')
print(f'Warmup steps: {warmup_steps}')

Steps/epoch : 597
Total steps : 10149
Warmup steps: 2029


In [9]:
for epoch in range(1, cfg.NUM_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{cfg.NUM_EPOCHS}', leave=False):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(input_ids      = batch['input_ids'],
                       attention_mask = batch['attention_mask']).logits
        loss   = loss_fn(logits, batch['labels'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f'Epoch {epoch:02d}/{cfg.NUM_EPOCHS}  train_loss={total_loss/len(train_loader):.4f}')

print('\nTraining complete.')

Epoch 01/17:   0%|          | 0/597 [00:00<?, ?it/s]

Epoch 01/17  train_loss=1.1451


Epoch 02/17:   0%|          | 0/597 [00:00<?, ?it/s]

Epoch 02/17  train_loss=0.5879


Epoch 03/17:   0%|          | 0/597 [00:00<?, ?it/s]

...

Epoch 15/17  train_loss=0.2739


Epoch 16/17:   0%|          | 0/597 [00:00<?, ?it/s]

Epoch 16/17  train_loss=0.2678


Epoch 17/17:   0%|          | 0/597 [00:00<?, ?it/s]

Epoch 17/17  train_loss=0.2684

Training complete.


<div id = "6-predict" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">6. Test Set Predictions</h2>
</div>

In [10]:
test_ds     = TweetDataset(x_test, labels=None, tokenizer=tokenizer, max_length=cfg.MAX_LENGTH)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE,
                         shuffle=False, num_workers=2)

model.eval()
test_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Predicting test set'):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        preds = model(input_ids      = batch['input_ids'],
                      attention_mask = batch['attention_mask']).logits.argmax(dim=-1)
        test_preds.extend(preds.cpu().numpy())

test_preds = np.array(test_preds)
print('Prediction distribution:')
print(pd.Series(test_preds).map(LABEL_NAMES).value_counts().to_dict())

Predicting test set:   0%|          | 0/150 [00:00<?, ?it/s]

Prediction distribution:
{'Neutral': 1527, 'Bullish': 489, 'Bearish': 372}


<div id = "7-save" style="background:#1e293b; padding:16px 20px; border-radius:6px; margin-bottom:4px;">
<h2 style="color:#f8fafc; margin:0;">7. Save Predictions</h2>
</div>

In [11]:
pred_df = pd.DataFrame({'id': test_ids, 'y_pred': test_preds})
pred_df.to_csv('pred_24.csv', index=False)

print(f'Saved -> pred_24.csv')
print(f'Rows  : {len(pred_df):,}')
print(f'Cols  : {list(pred_df.columns)}')
print()
print(pred_df.head())

Saved -> pred_24.csv
Rows  : 2,388
Cols  : ['id', 'y_pred']

   id  y_pred
0   0       2
1   1       2
2   2       2
3   3       2
4   4       2
